# Notebook 4 — Factor Research Library

Visualises the pre-computed factor IC/ICIR outputs from `scripts/factor_research.py`.

**Requires:** `reports/factor_research_{1y,3y,5y}_sn.csv`  
Run `python3 scripts/factor_research.py --all-horizons` first if those files are missing.

**Sections:**
1. Top factors by |ICIR| — all three horizons
2. IC decay across horizons (1y → 3y → 5y) — optimal holding period signal
3. IC per year heatmap — consistency / regime analysis
4. Factor turnover — ranking stability (cost of trading each signal)
5. Fraud vs Value signal comparison
6. Factor IC correlation across horizons — which signals agree?

## Setup — run this first (works in Google Colab and locally)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats as scipy_stats

plt.style.use('dark_background')

# ── Detect environment ────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # !pip install huggingface_hub --quiet
    from huggingface_hub import hf_hub_download
    HF_REPO = os.environ.get('HF_REPO', 'your-username/stock-screener-data')
    token   = os.environ.get('HF_TOKEN')
    parquet_path = hf_hub_download(
        repo_id=HF_REPO, filename='historical_dataset_clean.parquet',
        repo_type='dataset', token=token,
    )
    REPORTS = Path('reports')
    REPORTS.mkdir(exist_ok=True)
    for fname in ['factor_research_1y_sn.csv', 'factor_research_3y_sn.csv',
                  'factor_research_5y_sn.csv']:
        hf_hub_download(repo_id=HF_REPO, filename=f'reports/{fname}',
                        repo_type='dataset', token=token, local_dir='.')
else:
    parquet_path = '../data/historical_dataset_clean.parquet'
    REPORTS      = Path('../reports')

# ── Load dataset (annual rows only) ──────────────────────────────────────────
df_all = pd.read_parquet(parquet_path)
annual = df_all[df_all['period_type'] == 'annual'].copy()
print(f"Loaded {len(annual):,} annual rows | "
      f"{annual['ticker'].nunique():,} companies | "
      f"{annual['fiscal_year'].nunique()} years")

# ── Load factor research CSVs ─────────────────────────────────────────────────
HORIZONS = ['1y', '3y', '5y']
RET_COLS = {'1y': 'forward_return_1y', '3y': 'forward_return_3y', '5y': 'forward_return_5y'}

fr = {}
for h in HORIZONS:
    fp = REPORTS / f'factor_research_{h}_sn.csv'
    if fp.exists():
        fr[h] = pd.read_csv(fp)
        print(f"  {h}: {len(fr[h])} factors | "
              f"sig (|t|≥2): {(fr[h]['ic_tstat'].abs() >= 2).sum()}")
    else:
        # Fallback: try without sector-neutral suffix
        fp2 = REPORTS / f'factor_research_{h}.csv'
        if fp2.exists():
            fr[h] = pd.read_csv(fp2)
            print(f"  {h}: {len(fr[h])} factors (non-SN fallback)")
        else:
            print(f"  {h}: NOT FOUND — run: python3 scripts/factor_research.py --all-horizons")

if not fr:
    raise RuntimeError("No factor research CSVs found. Run factor_research.py first.")

## 1. Top factors by |ICIR|

ICIR = Mean IC / StdIC — measures signal quality (signal-to-noise ratio).  
Rule of thumb: |ICIR| > 0.3 = strong, > 0.1 = useful, < 0.05 = noise.

In [ ]:
n_plots = len(fr)
fig, axes = plt.subplots(1, n_plots, figsize=(9 * n_plots, 10))
if n_plots == 1:
    axes = [axes]

for ax, h in zip(axes, [h for h in HORIZONS if h in fr]):
    df_h = fr[h].copy()
    df_h['abs_icir'] = df_h['icir'].abs()
    top = df_h.nlargest(20, 'abs_icir').sort_values('icir')
    c = ['#2ecc71' if v > 0 else '#e74c3c' for v in top['icir']]
    ax.barh(top['feature'], top['icir'], color=c, alpha=0.85)
    ax.axvline(0, color='white', alpha=0.3)
    # Mark statistically significant bars
    for i, (_, row) in enumerate(top.iterrows()):
        if abs(row['ic_tstat']) >= 2.0:
            ax.text(row['icir'] + (0.002 if row['icir'] >= 0 else -0.002),
                    i, '★', va='center', fontsize=8,
                    ha='left' if row['icir'] >= 0 else 'right', color='yellow')
    ax.set_title(f'Top 20 Factors by |ICIR| ({h} horizon)')
    ax.set_xlabel('ICIR  (★ = |t| ≥ 2)')

plt.tight_layout()
plt.show()

# Summary table
for h in [h for h in HORIZONS if h in fr]:
    df_h = fr[h].copy()
    df_h['abs_icir'] = df_h['icir'].abs()
    print(f"\n── {h} — top 15 by |ICIR| ──────────────────")
    cols = ['feature', 'mean_ic', 'icir', 'ic_tstat', 'pct_positive_ic', 'n_years', 'turnover']
    cols = [c for c in cols if c in df_h.columns]
    print(df_h.nlargest(15, 'abs_icir')[cols].to_string(index=False))

## 2. IC decay across horizons (1y → 3y → 5y)

How does a factor's signal strength change as the holding period grows?  
- Factor IC rises 1y→5y: slow-burn structural signal (value, quality)  
- Factor IC falls 1y→5y: momentum/short-lived signal  
- Consistent sign across all horizons = robust, not regime-specific

In [ ]:
if len(fr) < 2:
    print("Need at least 2 horizons for decay analysis.")
else:
    # Merge all available horizons on feature name
    base = fr['1y'][['feature', 'mean_ic', 'icir', 'ic_tstat']].rename(
        columns={'mean_ic': 'ic_1y', 'icir': 'icir_1y', 'ic_tstat': 'tstat_1y'})
    merged = base.copy()
    for h in ['3y', '5y']:
        if h in fr:
            tmp = fr[h][['feature', 'mean_ic', 'icir']].rename(
                columns={'mean_ic': f'ic_{h}', 'icir': f'icir_{h}'})
            merged = merged.merge(tmp, on='feature', how='inner')

    ic_cols = [c for c in ['ic_1y', 'ic_3y', 'ic_5y'] if c in merged.columns]
    x_labels = [c.replace('ic_', '') + ' horizon' for c in ic_cols]
    x_vals   = list(range(len(ic_cols)))

    # Select top 12 by |icir_1y| + bottom 5 by icir_1y (most negative)
    merged['abs_icir_1y'] = merged['icir_1y'].abs()
    top12  = merged.nlargest(12, 'abs_icir_1y')['feature'].tolist()
    bot5   = merged.nsmallest(5, 'icir_1y')['feature'].tolist()
    plot_feats = list(dict.fromkeys(top12 + bot5))  # deduplicate, preserve order
    plot_df = merged[merged['feature'].isin(plot_feats)].copy()

    fig, ax = plt.subplots(figsize=(11, 6))
    cmap = plt.cm.tab20
    for i, (_, row) in enumerate(plot_df.iterrows()):
        ics = [row[c] for c in ic_cols]
        is_top = row['feature'] in top12
        alpha  = 0.9 if is_top else 0.45
        lw     = 2.0 if is_top else 1.0
        ax.plot(x_vals, ics, marker='o', color=cmap(i / len(plot_df)),
                alpha=alpha, lw=lw, label=row['feature'])

    ax.axhline(0, color='white', linestyle='--', alpha=0.3)
    ax.set_xticks(x_vals)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel('Mean IC (Spearman rank correlation with forward return)')
    ax.set_title('IC Decay Across Investment Horizons — Top/Bottom Factors')
    ax.legend(fontsize=7, loc='center right', ncol=2, framealpha=0.3)
    plt.tight_layout()
    plt.show()

    # Consistent-sign factors — robust across all horizons
    all_pos = merged[ic_cols].gt(0).all(axis=1)
    all_neg = merged[ic_cols].lt(0).all(axis=1)
    consistent = merged[all_pos | all_neg].copy().sort_values('abs_icir_1y', ascending=False)
    print(f"\nFactors with consistent IC sign across all {len(ic_cols)} horizons: {len(consistent)}")
    print(consistent[['feature'] + ic_cols + ['tstat_1y']].head(20).to_string(index=False))

## 3. IC per year — consistency / regime check

A factor that only works in bull markets is not a reliable signal.  
Blue = positive IC (factor predicts higher returns that year).  
Red = negative IC (factor goes against returns that year — regime flip).

In [ ]:
def _ic_per_year(df, feature, ret_col):
    sub = df[df[ret_col].notna() & df[feature].notna()]
    out = {}
    for yr in sorted(sub['fiscal_year'].unique()):
        grp = sub[sub['fiscal_year'] == yr]
        if len(grp) < 30:
            continue
        c, _ = scipy_stats.spearmanr(grp[feature], grp[ret_col])
        if not np.isnan(c):
            out[yr] = round(c, 4)
    return out

ret_col = RET_COLS['1y']
if '1y' in fr and ret_col in annual.columns:
    df_h = fr['1y'].copy()
    df_h['abs_icir'] = df_h['icir'].abs()
    top15_feats = [
        f for f in df_h.nlargest(15, 'abs_icir')['feature'].tolist()
        if f in annual.columns
    ]

    print(f"Recomputing per-year IC for top {len(top15_feats)} factors...")
    ic_matrix = {}
    for feat in top15_feats:
        ic_matrix[feat] = _ic_per_year(annual, feat, ret_col)

    ic_df = pd.DataFrame(ic_matrix).T.sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(14, 6))
    vmax = max(0.10, ic_df.abs().max().max())
    im = ax.imshow(ic_df.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, label='IC (Spearman)')
    ax.set_xticks(range(len(ic_df.columns)))
    ax.set_xticklabels(ic_df.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(ic_df.index)))
    ax.set_yticklabels(ic_df.index, fontsize=9)
    ax.set_title('IC Per Year Heatmap — Top 15 Factors (1y horizon)')
    plt.tight_layout()
    plt.show()

    # Consistency stats
    consistency = pd.DataFrame({
        'mean_ic':    ic_df.mean(axis=1).round(4),
        'std_ic':     ic_df.std(axis=1).round(4),
        'pct_pos':    ic_df.gt(0).mean(axis=1).round(3),
        'worst_year': ic_df.min(axis=1).round(4),
        'best_year':  ic_df.max(axis=1).round(4),
    }).sort_values('mean_ic', ascending=False)
    print("\nIC per-year consistency:")
    print(consistency.to_string())
else:
    print(f"Skipping: '1y' factor data or '{ret_col}' column not available.")

## 4. Factor turnover — rank stability (trading cost proxy)

Turnover = mean Spearman correlation of factor rankings between consecutive years.  
- High correlation (near 1.0) = rankings barely change year to year = **low turnover = cheap to trade**  
- Low correlation (near 0) = rankings shuffle dramatically = **high turnover = expensive**  

Best factors: high |ICIR| AND high rank stability.

In [ ]:
if '1y' in fr:
    tv = fr['1y'][fr['1y']['turnover'].notna()].sort_values('turnover').copy()
    tv['abs_icir'] = tv['icir'].abs()

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    low = tv.head(15)
    axes[0].barh(low['feature'], low['turnover'], color='#2ecc71', alpha=0.8)
    axes[0].set_title('Lowest Turnover — Stable Rankings (cheapest to trade)')
    axes[0].set_xlabel('Consecutive-year rank correlation')

    high = tv.tail(15).sort_values('turnover', ascending=False)
    axes[1].barh(high['feature'], high['turnover'], color='#e74c3c', alpha=0.8)
    axes[1].set_title('Highest Turnover — Unstable Rankings (most expensive)')
    axes[1].set_xlabel('Consecutive-year rank correlation')

    plt.suptitle('Factor Turnover — Rank Stability Year over Year', y=1.02)
    plt.tight_layout()
    plt.show()

    # Sweet spot: high |ICIR| + low turnover
    tv_med = tv['turnover'].median()
    sweet  = tv[(tv['abs_icir'] >= 0.10) & (tv['turnover'] <= tv_med)].sort_values('abs_icir', ascending=False)
    print(f"Sweet-spot factors (|ICIR|≥0.10 AND turnover below median {tv_med:.3f}): {len(sweet)}")
    cols = ['feature', 'mean_ic', 'icir', 'ic_tstat', 'turnover']
    cols = [c for c in cols if c in sweet.columns]
    print(sweet[cols].head(20).to_string(index=False))

## 5. Fraud vs Value signal comparison

Core thesis: fraud signals should predict **lower** returns (negative IC).  
Value signals should predict **higher** returns (positive IC).

In [ ]:
FRAUD_FEATURES = [
    'beneish_score', 'beneish_flag', 'piotroski_f_score', 'piotroski_score',
    'accruals_ratio', 'cfd_ratio', 'altman_score', 'ar_ratio', 'non_op_ratio',
    'revenue_quality_flag', 'earnings_quality_flag', 'accruals_flag', 'cfd_flag',
]
VALUE_FEATURES = [
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'fcf_yield', 'ev_ebitda', 'pb_ratio', 'pe_ratio',
    'ncav_ratio', 'croic', 'roe', 'roa', 'gross_margin', 'net_margin',
]

if '1y' in fr:
    df_h = fr['1y'].copy()
    df_h['type'] = 'other'
    df_h.loc[df_h['feature'].isin(FRAUD_FEATURES), 'type'] = 'fraud'
    df_h.loc[df_h['feature'].isin(VALUE_FEATURES), 'type'] = 'value'

    classified = df_h[df_h['type'] != 'other'].copy()

    if len(classified) == 0:
        print("None of the named fraud/value features found in factor research output.")
        print("Available features (first 20):", df_h['feature'].head(20).tolist())
    else:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        type_color = {'fraud': '#e74c3c', 'value': '#2ecc71'}

        for ax, sig_type in zip(axes, ['fraud', 'value']):
            grp = classified[classified['type'] == sig_type].sort_values('icir')
            if len(grp) == 0:
                ax.set_title(f'No {sig_type} features found')
                continue
            bar_c = ['#2ecc71' if v > 0 else '#e74c3c' for v in grp['icir']]
            ax.barh(grp['feature'], grp['icir'], color=bar_c, alpha=0.85)
            ax.axvline(0, color='white', alpha=0.3)
            ax.set_title(f'{sig_type.title()} Signals — ICIR (1y)')
            ax.set_xlabel('ICIR')

        plt.suptitle('Fraud vs Value Signals — Factor Alpha (1y horizon, sector-neutral)', y=1.02)
        plt.tight_layout()
        plt.show()

        print("\nFraud signals (sorted by ICIR):")
        frd = classified[classified['type'] == 'fraud']
        cols = ['feature', 'mean_ic', 'icir', 'ic_tstat', 'pct_positive_ic', 'n_years']
        cols = [c for c in cols if c in frd.columns]
        print(frd.sort_values('icir')[cols].to_string(index=False))

        print("\nValue signals (sorted by ICIR):")
        val = classified[classified['type'] == 'value']
        print(val.sort_values('icir', ascending=False)[cols].to_string(index=False))

## 6. Factor IC correlation across horizons

Do factors that work at 1y also work at 3y and 5y?  
High correlation → the signal is structural (persistent).  
Low/negative correlation → short-term vs long-term signals diverge.

In [ ]:
if 'merged' not in dir() or len(fr) < 2:
    # Rebuild merged from fr
    if '1y' in fr:
        merged = fr['1y'][['feature', 'mean_ic', 'icir', 'ic_tstat']].rename(
            columns={'mean_ic': 'ic_1y', 'icir': 'icir_1y', 'ic_tstat': 'tstat_1y'})
        for h in ['3y', '5y']:
            if h in fr:
                tmp = fr[h][['feature', 'mean_ic', 'icir']].rename(
                    columns={'mean_ic': f'ic_{h}', 'icir': f'icir_{h}'})
                merged = merged.merge(tmp, on='feature', how='inner')
        merged['abs_icir_1y'] = merged['icir_1y'].abs()

ic_cols = [c for c in ['ic_1y', 'ic_3y', 'ic_5y'] if c in merged.columns]

if len(ic_cols) >= 2:
    pairs = [(ic_cols[0], ic_cols[i]) for i in range(1, len(ic_cols))]
    fig, axes = plt.subplots(1, len(pairs), figsize=(8 * len(pairs), 6))
    if len(pairs) == 1:
        axes = [axes]

    for ax, (col_a, col_b) in zip(axes, pairs):
        sub = merged[[col_a, col_b, 'feature', 'abs_icir_1y']].dropna()
        # Colour by absolute ICIR: brighter = stronger signal
        sizes = np.clip(sub['abs_icir_1y'] * 200, 10, 120)
        sc = ax.scatter(sub[col_a], sub[col_b], s=sizes,
                        c=sub['abs_icir_1y'], cmap='YlOrRd', alpha=0.7, edgecolors='none')
        plt.colorbar(sc, ax=ax, label='|ICIR| 1y')

        # Label top outliers by distance from origin
        sub['dist'] = np.sqrt(sub[col_a]**2 + sub[col_b]**2)
        for _, row in sub.nlargest(8, 'dist').iterrows():
            ax.annotate(row['feature'], (row[col_a], row[col_b]),
                        fontsize=7, alpha=0.8, ha='left')

        # Regression line
        m, b, r, p, _ = scipy_stats.linregress(sub[col_a], sub[col_b])
        x_line = np.linspace(sub[col_a].min(), sub[col_a].max(), 50)
        ax.plot(x_line, m * x_line + b, color='#e74c3c', lw=1.5, alpha=0.8,
                label=f'r = {r:.2f}  (p={p:.3f})')

        ax.axhline(0, color='white', alpha=0.2)
        ax.axvline(0, color='white', alpha=0.2)
        ax.set_xlabel(col_a.replace('ic_', 'Mean IC '), fontsize=11)
        ax.set_ylabel(col_b.replace('ic_', 'Mean IC '), fontsize=11)
        ax.set_title(f'IC: {col_a} vs {col_b}')
        ax.legend(fontsize=9)

    plt.suptitle('Factor IC Correlation Across Horizons\n'
                 '(dot size = |ICIR|; upper-right = strong positive at both horizons)', y=1.02)
    plt.tight_layout()
    plt.show()

    print("\nIC correlation matrix across horizons:")
    print(merged[ic_cols].corr().round(3))

    # Factors in upper-right quadrant (positive IC at all horizons)
    upper_right = merged[(merged[ic_cols] > 0).all(axis=1)].sort_values('abs_icir_1y', ascending=False)
    print(f"\nFactors with positive IC at ALL horizons: {len(upper_right)}")
    print(upper_right[['feature'] + ic_cols].head(15).to_string(index=False))
else:
    print("Need at least 2 horizons for cross-horizon correlation.")